<a href="https://colab.research.google.com/github/ganapathyhari/RAG/blob/main/RAG_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Imports**

In [10]:
import numpy as np
import hashlib
from typing import List, Tuple, Dict


**rag/chunking**

In [4]:
from typing import List

def paragraph_chunk(text: str) -> List[str]:
    """
    Simple paragraph-based chunker.
    Splits on double newlines and removes empty chunks.
    """
    if not isinstance(text, str):
        raise TypeError("Input text must be a string")

    chunks = [p.strip() for p in text.split("\n\n") if p.strip()]
    return chunks


**rag/embeddings**

In [5]:
from typing import List
import numpy as np
import hashlib

class MockEmbeddingModel:
    """
    Deterministic pseudo-embedding using hashing.
    Good for demos without external API calls.
    """

    def embed(self, text: str) -> np.ndarray:
        h = hashlib.sha256(text.encode()).digest()
        arr = np.frombuffer(h, dtype=np.uint8).astype(float)
        return arr / np.linalg.norm(arr)


**rag/vector_store**

In [6]:
import numpy as np
from typing import Dict, List, Tuple

class InMemoryVectorStore:
    def __init__(self):
        self.store: Dict[str, np.ndarray] = {}

    def add(self, doc_id: str, vector: np.ndarray):
        self.store[doc_id] = vector

    def all(self) -> List[Tuple[str, np.ndarray]]:
        return list(self.store.items())


**rag/retrieval**

In [7]:
import numpy as np
from typing import List, Tuple

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    if np.linalg.norm(a) == 0 or np.linalg.norm(b) == 0:
        return 0.0
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


def top_k(query_vec: np.ndarray, vectors: List[Tuple[str, np.ndarray]], k: int = 3):
    scores = []

    for doc_id, vec in vectors:
        score = cosine_similarity(query_vec, vec)
        scores.append((score, doc_id))

    scores.sort(reverse=True)
    return scores[:k]


**rag/metrics**

In [8]:
from typing import List
from collections import Counter

def f1_score(true_tokens: List[str], pred_tokens: List[str]) -> float:
    true_set = set(true_tokens)
    pred_set = set(pred_tokens)

    tp = len(true_set & pred_set)
    fp = len(pred_set - true_set)
    fn = len(true_set - pred_set)

    if tp == 0:
        return 0.0

    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    return 2 * (precision * recall) / (precision * recall + precision + recall)


def rouge_1(true_text: str, pred_text: str) -> float:
    true_tokens = true_text.split()
    pred_tokens = pred_text.split()

    true_count = Counter(true_tokens)
    pred_count = Counter(pred_tokens)

    overlap = sum((true_count & pred_count).values())
    total = sum(true_count.values())

    return overlap / total if total > 0 else 0.0


**rag/rag_pipeline**

In [11]:
class RAGPipeline:
    def __init__(self):
        self.embedder = MockEmbeddingModel()
        self.store = InMemoryVectorStore()

    def ingest(self, documents: List[str]):
        for idx, doc in enumerate(documents):
            for chunk in paragraph_chunk(doc):
                vec = self.embedder.embed(chunk)
                chunk_id = f"doc{idx}_chunk{hash(chunk)}"
                self.store.add(chunk_id, vec)

    def query(self, query: str, documents: List[str], k: int = 3):
        self.ingest(documents)

        query_vec = self.embedder.embed(query)
        candidates = top_k(query_vec, self.store.all(), k)

        top_chunks = [doc_id for _, doc_id in candidates]
        answer = "Relevant chunks: " + ", ".join(top_chunks)

        return {"answer": answer, "top_chunks": top_chunks}


**Test it**

In [12]:
docs = [
    "Atlassian builds Jira, Confluence and Trello.\n\nUsed by millions worldwide.",
    "RAG pipelines enhance LLM accuracy by grounding responses in knowledge bases."
]

rag = RAGPipeline()
result = rag.query("What does Atlassian build?", documents=docs)

result


{'answer': 'Relevant chunks: doc0_chunk-2568378801537353518, doc0_chunk1068737947919680526, doc1_chunk-338618499611087718',
 'top_chunks': ['doc0_chunk-2568378801537353518',
  'doc0_chunk1068737947919680526',
  'doc1_chunk-338618499611087718']}